# 2 · Reading documents

⏱ about 7 minutes

To answer from the papers, the bot first has to read them. A PDF only stores where to draw each
letter, not what is a heading, a paragraph or a table.
[Docling](https://github.com/docling-project/docling) looks at every page with a layout model,
works out its structure and writes it out as Markdown.

![Docling finds headings, tables and figures on a PDF page and writes Markdown](img/docling.svg)

We convert page by page and keep the file name and page number with every page; notebook 4
needs them for the citations. This takes about a minute.

In [ ]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# These PDFs contain real text, so we skip OCR (reading text out of images).
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=PdfPipelineOptions(do_ocr=False))}
)

pages = []
for pdf in sorted(Path("data").glob("*.pdf")):
    document = converter.convert(pdf).document
    for number in document.pages:
        pages.append({"source": pdf.name, "page": number, "text": document.export_to_markdown(page_no=number)})

print(len(pages), "pages")

Here is page 3 of the Schmidt paper as Docling sees it. The table came through as a table.

In [ ]:
from IPython.display import Markdown

page = next(p for p in pages if p["source"].startswith("Schmidt") and p["page"] == 3)
Markdown(page["text"])

## Chunks

A whole page is too much text to hand the model for every question, and too coarse to find the
one paragraph that matters. So we cut every page into **chunks** of about 1000 characters.
Neighbouring chunks overlap by 200 characters, so a sentence cut at a border still appears whole
in one of them.

![A page cut into three overlapping chunks](img/chunking.svg)

In [ ]:
def chunk(text, size=1000, overlap=200):
    starts = range(0, max(len(text) - overlap, 1), size - overlap)
    return [text[start : start + size] for start in starts]


chunks = [
    {"source": p["source"], "page": p["page"], "text": piece}
    for p in pages
    for piece in chunk(p["text"])
    if piece.strip()
]
print(len(chunks), "chunks\n")
print(chunks[10]["text"])

### Look closely: where does this chunk start and end?

Each chunk is 1000 characters long and a new one starts every 800, so neighbours share 200. Docling
knows where every heading and table is, but our chunker ignores that, and cuts through a table if
one is in the way. Better chunkers cut along the document's structure.

Notebook 3 continues with these chunks, so we save them.

In [ ]:
import json

Path("chunks.json").write_text(json.dumps(chunks))
print("saved", len(chunks), "chunks to chunks.json")

## A document inspector in Chainlit

This bot reads any PDF you attach to a message. Attached files arrive in `message.elements`.
Docling needs a few seconds per document; `cl.make_async` runs it in the background so the chat
stays responsive, and `cl.Step` shows it as a step you can unfold.

In [ ]:
%%writefile app.py
import chainlit as cl
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# These PDFs contain real text, so we skip OCR (reading text out of images).
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=PdfPipelineOptions(do_ocr=False))}
)


def chunk(text, size=1000, overlap=200):
    starts = range(0, max(len(text) - overlap, 1), size - overlap)
    return [text[start : start + size] for start in starts]


@cl.on_chat_start
async def on_chat_start():
    await cl.Message(content="Attach a PDF (paperclip icon, or drag it in) and I'll show you what Docling reads.").send()


@cl.on_message
async def on_message(message: cl.Message):
    pdfs = [file for file in message.elements if file.name.lower().endswith(".pdf")]
    if not pdfs:
        await cl.Message(content="Please attach a PDF.").send()
        return

    for pdf in pdfs:
        async with cl.Step(name=f"Docling reads {pdf.name}"):
            document = (await cl.make_async(converter.convert)(pdf.path)).document
        pages = [document.export_to_markdown(page_no=number) for number in document.pages]
        chunks = [piece for text in pages for piece in chunk(text) if piece.strip()]

        await cl.Message(
            content=f"**{pdf.name}**: {len(pages)} pages, {len(chunks)} chunks. This is page 1:\n\n---\n\n{pages[0]}"
        ).send()
        # TODO 1: send the first three chunks, each as its own message.
        # Bonus: send every table Docling found.

In [ ]:
import socket
import subprocess
import sys

# Start the chatbot, unless it's still running from an earlier notebook.
with socket.socket() as probe:
    running = probe.connect_ex(("127.0.0.1", 8000)) == 0
if not running:
    subprocess.Popen(
        [sys.executable, "-m", "chainlit", "run", "app.py", "--watch", "--headless"],
        stdout=open("chainlit.log", "w"),
        stderr=subprocess.STDOUT,
    )
print("Chatbot: http://localhost:8000")

## Your turn

Edit the `%%writefile app.py` cell above, run it again, and try your change in the browser.

Open **http://localhost:8000** and attach a PDF: one from the `data/` folder, or one of your own.

1. **Where are the cuts?** Send the first three chunks, each as its own message. Then try
   `size=300` and `size=3000` in `chunk()`. How does the number of chunks change? Which size
   would you want for answering questions?

**Bonus: only the tables.** `document.tables` is the list of tables Docling found, and
`table.export_to_markdown(doc=document)` turns one into Markdown. Send each table as its own
message. The Schmidt paper has one, the Lin paper three.

<details>
<summary><b>Solutions</b> (try first!)</summary>

**1**
```python
        for piece in chunks[:3]:
            await cl.Message(content=piece).send()
```

**Bonus**
```python
        for table in document.tables:
            await cl.Message(content=table.export_to_markdown(doc=document)).send()
```
</details>